In [1]:
##Dev tfmodisco thru FIMO
import h5py
import torch
import pandas as pd
import numpy as np
from tangermeme.utils import characters
from tangermeme.seqlet import tfmodisco_seqlets
from tangermeme.plot import plot_logo
from tangermeme.utils import one_hot_encode
from tangermeme.deep_lift_shap import deep_lift_shap, _nonlinear
import torch, torch.nn as nn, torch.fx as fx, copy
import copy
import matplotlib.pyplot as plt
import statsmodels
import modiscolite

In [2]:
#get one hots

def load_attributions_with_test_index(h5_file_path):
    """
    Load scaled foreground attributions while preserving original sequence indices
    """
    with h5py.File(h5_file_path, 'r') as f:
        orig_df = pd.DataFrame(pd.read_csv('/grid/wsbs/home_norepl/pmantill/Motif_Swap_Experiments/Replicating_Motif_Swap_Manuscript/experimental_library_generation/Binned_libraries/Dev_final_df.csv'))
        # Load the attribution data (shape: 8999, 3, 249, 4)
        all_attributions = f['attributions'][:]
        print(all_attributions.shape)
        
        # Load the original indices mapping (shape: 8999,)
        #original_indices = f['original_indices'][:]
        
        # Extract scaled foreground (index 1)
        scaled_foreground_attr = torch.tensor(all_attributions[:, 1, :, :], dtype=torch.float32)
        cluster_foreground_attr = torch.tensor(all_attributions[:, 0, :, :], dtype=torch.float32)
        average_background_attr = torch.tensor(all_attributions[:, 2, :, :], dtype=torch.float32)
        test_idx = orig_df["Unnamed: 0"]
        bin = orig_df["EvoAug_Score_dev_bin"]
        real_score = orig_df["Real_Score_dev"]
        pred = orig_df["EvoAug_Score_dev"]
        seq = orig_df["Sequence"]

        df = pd.DataFrame(list(zip(scaled_foreground_attr, cluster_foreground_attr, average_background_attr, test_idx, bin, real_score, pred, seq)), columns=['Scaled_foreground', 'Cluster_foreground', 'Avg_background', 'test_idx', 'bin', 'real_score', 'evoaug_pred', "str_seq"])
        
        print(df.shape)
        
        return df

h5_file_path = '/grid/wsbs/home_norepl/pmantill/Motif_Swap_Experiments/Replicating_Motif_Swap_Manuscript/SEAM_analysis/Dev_SEAM_results/Dev_task0_attributions.h5'

df = load_attributions_with_test_index(h5_file_path)



(9000, 3, 249, 4)
(9000, 8)


In [3]:
df["ohe_seq"] = None
ohe_seqs = []
for i in range(len(df)):
    ohe_seq = one_hot_encode(df.iloc[i]["str_seq"])
    ohe_seqs.append(ohe_seq)

df["ohe_seq"] = ohe_seqs
    


In [4]:
# split by bin 

low_df = df[df["bin"] == 'Low'].reset_index()
mid_df = df[df["bin"] == 'Mid'].reset_index()
high_df = df[df["bin"] == 'High'].reset_index()

print(len(low_df), len(mid_df), len(high_df))

low_ohe_list = low_df["ohe_seq"].values          # array of length N, each entry shape (L, 4)
low_att_list = low_df["Scaled_foreground"].values

mid_ohe_list = mid_df["ohe_seq"].values
mid_att_list = mid_df["Scaled_foreground"].values

high_ohe_list = high_df["ohe_seq"].values
high_att_list = high_df["Scaled_foreground"].values

# If entries are torch tensors, convert each to numpy float32
def to_numpy32(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy().astype('float32')
    x = np.asarray(x)
    return x.astype('float32')

low_ohe = np.stack([to_numpy32(x) for x in low_ohe_list], axis=0).transpose(0, 2, 1)   # shape (N, L, 4)
low_att = np.stack([to_numpy32(x) for x in low_att_list], axis=0)   # shape (N, L, 4)

mid_ohe = np.stack([to_numpy32(x) for x in mid_ohe_list], axis=0).transpose(0, 2, 1) 
mid_att = np.stack([to_numpy32(x) for x in mid_att_list], axis=0)

high_ohe = np.stack([to_numpy32(x) for x in high_ohe_list], axis=0).transpose(0, 2, 1) 
high_att = np.stack([to_numpy32(x) for x in high_att_list], axis=0)

print(low_ohe.shape, high_att.shape)  # Expect (N_low, L, 4) and (N_high, L, 4)

3000 3000 3000
(3000, 249, 4) (3000, 249, 4)


In [41]:
low_df.to_csv("../experimental_library_generation/Binned_libraries/Dev_full_data/low_df.csv", index=True)
mid_df.to_csv("../experimental_library_generation/Binned_libraries/Dev_full_data/mid_df.csv", index=True)
high_df.to_csv("../experimental_library_generation/Binned_libraries/Dev_full_data/high_df.csv", index=True)


In [5]:
import modiscolite
import numpy as np

def run_bin(ohe, att, window_size=20, flank=5):
    assert ohe.shape == att.shape and ohe.shape[-1] == 4
    ohe = ohe.astype('float32')
    att = att.astype('float32')
    pos_patterns, neg_patterns = modiscolite.tfmodisco.TFMoDISco(
        one_hot=ohe,
        hypothetical_contribs=att,
        sliding_window_size=window_size,
        flank_size=flank,
        initial_flank_to_add=0,
        verbose=True
    )
    return pos_patterns, neg_patterns

pos_low, neg_low = run_bin(low_ohe, low_att)   # (N_low, L, 4)
pos_mid, neg_mid = run_bin(mid_ohe, mid_att)   # (N_mid, L, 4)
pos_high, neg_high = run_bin(high_ohe, high_att)  # (N_high, L, 4)

Using 1339 positive seqlets
Extracted 2013 negative seqlets
Using 4845 positive seqlets
Extracted 1075 negative seqlets
Using 8870 positive seqlets


In [6]:
# Save each bin separately
modiscolite.io.save_hdf5('TF-Modisco-lite_results/h5py_files/Dev_low_bin_motifs.h5', pos_low, neg_low, 30)
modiscolite.io.save_hdf5('TF-Modisco-lite_results/h5py_files/Dev_mid_bin_motifs.h5', pos_mid, neg_mid, 30)  
modiscolite.io.save_hdf5('TF-Modisco-lite_results/h5py_files/Dev_high_bin_motifs.h5', pos_high, neg_high, 30)


In [7]:
import os
import subprocess
from modiscolite.util import MemeDataType
import modiscolite


# Define bins and corresponding .h5 inputs
bins = {
    'low':  'TF-Modisco-lite_results/h5py_files/Dev_low_bin_motifs.h5',
    'mid':  'TF-Modisco-lite_results/h5py_files/Dev_mid_bin_motifs.h5',
    'high': 'TF-Modisco-lite_results/h5py_files/Dev_high_bin_motifs.h5'
}

# Base directories
report_dir = 'TF-Modisco-lite_results'
meme_dir = os.path.join(report_dir, 'Dev_meme_full_reports')
os.makedirs(meme_dir, exist_ok=True)

# Loop over bins and datatypes using command line
for bin_name, h5_file in bins.items():
    # Generate PFM MEME files
    pfm_file = os.path.join(meme_dir, f'Dev_{bin_name}_full_report')
    print(f'Writing PFMs for {bin_name}-bin → {pfm_file}')
    subprocess.run([
        'modisco', 'report',
        '-i', h5_file,
        '-o', pfm_file
    ], check=True)

print("MEME file generation complete!")

Writing PFMs for low-bin → TF-Modisco-lite_results/Dev_meme_full_reports/Dev_low_full_report
Saving motif files!
Report generated: TF-Modisco-lite_results/Dev_meme_full_reports/Dev_low_full_report/report.html
Writing PFMs for mid-bin → TF-Modisco-lite_results/Dev_meme_full_reports/Dev_mid_full_report
Saving motif files!
Report generated: TF-Modisco-lite_results/Dev_meme_full_reports/Dev_mid_full_report/report.html
Writing PFMs for high-bin → TF-Modisco-lite_results/Dev_meme_full_reports/Dev_high_full_report
Saving motif files!
Report generated: TF-Modisco-lite_results/Dev_meme_full_reports/Dev_high_full_report/report.html
MEME file generation complete!


In [32]:
## FIMO-lite
from memelite import fimo

motifs = 'TF-Modisco-lite_results/Dev_meme_full_reports/Dev_high_full_report/pfms/true_pfms.meme'

X = high_ohe.transpose(0, 2, 1)
print(X.shape)
print(len(motifs))

hits = fimo(motifs, X, threshold=0.001, dim=0)

(3000, 4, 249)
86
[[0.181658 0.180943 0.095168 ... 0.316647 0.374201 0.280676]
 [0.228835 0.22955  0.451136 ... 0.179956 0.18715  0.223122]
 [0.209535 0.250993 0.180229 ... 0.338229 0.230316 0.266287]
 [0.380372 0.338913 0.273867 ... 0.165568 0.208733 0.230316]]


In [33]:
hits[8]

,motif_name,motif_idx,sequence_name,start,end,strand,score,p-value
0,pos_patterns.pattern_16,8,6,155,185,+,5.113112,0.000351
1,pos_patterns.pattern_16,8,10,209,239,+,2.003794,0.000785
2,pos_patterns.pattern_16,8,10,217,247,+,2.105398,0.000767
3,pos_patterns.pattern_16,8,13,168,198,+,3.025889,0.000615
4,pos_patterns.pattern_16,8,15,130,160,+,7.148496,0.000189
...,...,...,...,...,...,...,...,...
1963,pos_patterns.pattern_16,8,2988,64,94,-,5.906746,0.000277
1964,pos_patterns.pattern_16,8,2989,57,87,-,5.160698,0.000351
1965,pos_patterns.pattern_16,8,2992,3,33,-,1.927561,0.000804
1966,pos_patterns.pattern_16,8,2995,37,67,-,2.437701,0.000713


In [34]:
import numpy as np
from statsmodels.stats.multitest import multipletests

def add_qvals_allow_na(hits,alpha, pcol='p-value', qcol='q-value'):
    for df in hits:
        if pcol not in df:
            # Create the column with NaNs so downstream code always sees it
            df[qcol] = np.nan
            continue

        p = df[pcol].to_numpy(dtype=float)
        mask = np.isfinite(p)  # True where p is valid
        q = np.full_like(p, np.nan, dtype=float)

        if mask.any():
            q_valid = multipletests(p[mask], alpha=alpha, method='fdr_bh')[1]
            q[mask] = q_valid

        df[qcol] = q
    return hits

# Usage:
hits = add_qvals_allow_na(hits, pcol='p-value', qcol='q-value', alpha=0.001)


In [35]:
q_filtered_hits = []
for df in hits:
    df = df[df["q-value"] < 0.001]
    q_filtered_hits.append(df)

import pandas as pd

combined_df_motifs = pd.concat(q_filtered_hits, ignore_index=True)


hits_byseq = dict(tuple(combined_df_motifs.groupby("sequence_name")))
len(combined_df_motifs)

54862

In [36]:
import pandas as pd

def collapse_motif_overlaps(df, same_thresh, diff_thresh):
    """
    Keep only the highest-scoring hit when two hits overlap with reciprocal overlap >= threshold.
    - Same motif_name: use same_thresh.
    - Different motif_name: use diff_thresh.
    Ignores strand; does not join/expand intervals.
    Required columns: start, end, motif_name, score.
    Optional: chrom.
    """
    required = {"start", "end", "motif_name", "score"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    hits = df.sort_values("score", ascending=False).to_dict("records")
    kept = []

    def reciprocal_overlap(a, b):
        # Require same chromosome if provided
        if "chrom" in a and "chrom" in b and a["chrom"] != b["chrom"]:
            return 0.0
        ov_start = max(a["start"], b["start"])
        ov_end   = min(a["end"],   b["end"])
        if ov_end <= ov_start:
            return 0.0
        ov = ov_end - ov_start
        la = max(1, a["end"] - a["start"])
        lb = max(1, b["end"] - b["start"])
        return ov / min(la, lb)

    while hits:
        seed = hits.pop(0)
        discard = False
        for i, kept_row in enumerate(kept):
            recip = reciprocal_overlap(seed, kept_row)
            if seed["motif_name"] == kept_row["motif_name"]:
                th = same_thresh
            else:
                th = diff_thresh
            if recip >= th:
                if seed["score"] > kept_row["score"]:
                    kept[i] = seed
                discard = True
                break
        if not discard:
            kept.append(seed)

    return pd.DataFrame(kept)

def collapse_motifs_byseq(motifs_byseq, same_thresh, diff_thresh):
    """
    Apply collapse_motif_overlaps to each sequence's DataFrame in motifs_byseq.
    motifs_byseq: dict[sequence_name] -> DataFrame
    Returns a single concatenated DataFrame with sequence_name preserved.
    """
    out_frames = []
    for seq_name, df in motifs_byseq.items():
        if "sequence_name" not in df.columns:
            df = df.copy()
            df["sequence_name"] = seq_name
        collapsed = collapse_motif_overlaps(df, same_thresh=same_thresh, diff_thresh=diff_thresh)
        out_frames.append(collapsed)
    return pd.concat(out_frames, ignore_index=True)

# Example usage:
new_hits = collapse_motifs_byseq(hits_byseq, same_thresh=0.60, diff_thresh=0.60)


In [37]:
import numpy as np
import pandas as pd

def add_test_idx_from_index(hits_byseq: dict[str, pd.DataFrame], df: pd.DataFrame, seq_col="sequence_name", out_col="test_idx"):
    """
    For each per-sequence DataFrame in hits_byseq, read its sequence_name and use that
    as an index lookup into high_df.index to fetch test_idx and assign as a new column.

    - hits_byseq: dict mapping sequence id/name -> DataFrame with a 'sequence_name' column.
    - high_df: DataFrame with 'test_idx' column, and index equal to sequence names.
    - seq_col: the column name in each per-sequence df containing its sequence label.
    - out_col: the column name to assign in each df (default 'test_idx').
    """
    if out_col not in df.columns:
        raise ValueError(f"'{out_col}' column not found in high_df")

    # Build lookup Series by index (sequence_name -> test_idx)
    lookup = df[out_col]  # index should be sequence_name

    updated = {}
    for key, df in hits_byseq.items():
        if seq_col not in df.columns:
            # If missing, infer from the dict key and set it for completeness
            seq_name = key
            g = df.copy()
            g[seq_col] = seq_name
        else:
            g = df
            # Prefer explicit column; ensure it's a single sequence per DF
            if g[seq_col].nunique() != 1:
                # If multiple sequence names somehow present, take the first row’s name for lookup
                seq_name = g[seq_col].iloc[0]
            else:
                seq_name = g[seq_col].iloc[0]

        # Lookup via index; NaN if not present
        val = lookup.reindex([seq_name]).iloc[0] if seq_name in lookup.index else np.nan
        g = g.copy()
        g[out_col] = val
        updated[key] = g

    return updated

# Example usage:
# hits_byseq = {seq: df, ...}  # each df has 'sequence_name'
# high_df must have index of sequence_name and a 'test_idx' column
hits_byseq = add_test_idx_from_index(hits_byseq, high_df, seq_col="sequence_name", out_col="test_idx")


In [38]:
hits_byseq

{0:                     motif_name  motif_idx  sequence_name  start  end strand  \
 2083    pos_patterns.pattern_0          0              0     40   70      -   
 2084    pos_patterns.pattern_0          0              0    104  134      -   
 2085    pos_patterns.pattern_0          0              0    200  230      -   
 19570  pos_patterns.pattern_14          6              0     86  116      -   
 27457   pos_patterns.pattern_2         11              0    166  196      +   
 27458   pos_patterns.pattern_2         11              0    204  234      +   
 29298   pos_patterns.pattern_2         11              0    110  140      -   
 31982   pos_patterns.pattern_3         12              0     19   49      -   
 34146   pos_patterns.pattern_5         14              0     68   98      +   
 35490   pos_patterns.pattern_5         14              0     86  116      -   
 38111   pos_patterns.pattern_6         15              0    145  175      -   
 42425   pos_patterns.pattern_7      

In [39]:
save_df = pd.concat(hits_byseq)
save_df.to_csv("TF-Modisco-lite_results/Dev_seq_hits/high_hits.csv", index=False)
